## Model Comparison For RHOMBUS, PRISM, CONE AND CUBE

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
# ===============================
# 🔧 Configuration
# ===============================
IMG_SIZE = 224
BATCH_SIZE = 32
TEST_DATASET_PATH = 'data/PlantVillage/test'  

In [ ]:
# ===============================
# 🔄 Data Augmentation
# ===============================

test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    TEST_DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False  # important to preserve label order
)

In [ ]:

# -----------------------------
# Function to evaluate a model
# -----------------------------
def evaluate_model(model, test_generator, model_name, codename):
    # Evaluate accuracy directly
    test_loss, test_acc, test_auc = model.evaluate(test_generator, verbose=1)

    # Predictions
    y_true = test_generator.classes
    y_pred_probs = model.predict(test_generator, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)

    # Classification report
    report = classification_report(
        y_true,
        y_pred,
        target_names=test_generator.class_indices.keys(),
        output_dict=True
    )

    precision = report['weighted avg']['precision']
    recall    = report['weighted avg']['recall']
    f1        = report['weighted avg']['f1-score']

    return {
        "Model": model_name,
        "Codename": codename,
        "Accuracy": round(test_acc, 4),
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1-Score": round(f1, 4),
        "AUC Score": round(test_auc, 4)
    }

In [ ]:
# -----------------------------
# Load trained models
# (Replace with your actual paths)
# -----------------------------
inception_model     = tf.keras.models.load_model("models/h5/best_cone_model.h5")
mobilenetv2_model   = tf.keras.models.load_model("models/h5/best_rhombus_model.h5")
mobilenetv2_cbam_model = tf.keras.models.load_model("models/h5/best_cube_model.h5")
xception_model      = tf.keras.models.load_model("models/h5/best_prism_model.h5")

In [ ]:
# -----------------------------
# Evaluate all models
# -----------------------------
results = []
results.append(evaluate_model(inception_model, test_generator, "Inception","CONE"))
results.append(evaluate_model(mobilenetv2_model, test_generator, "MobileNetV2","RHOMBUS"))
results.append(evaluate_model(mobilenetv2_cbam_model, test_generator, "MobileNetV2 + CBAM","CUBE"))
results.append(evaluate_model(xception_model, test_generator, "Xception","PRISM"))

In [ ]:
# -----------------------------
# Final Comparison Table
# -----------------------------
df_results = pd.DataFrame(results)
# Optional: save to CSV for thesis documentation
df_results.to_csv("evaluation_results/model_comparison_results.csv", index=False)
df_results